In [17]:
import io
import docx
import hashlib
import requests
import numpy as np
from datetime import datetime
from elasticsearch import Elasticsearch

In [99]:
def clean_line(line):
    line = line.strip()
    line = line.strip('\uFEFF')
    return line

def fetching_faq_document(document_id):
    url = f'https://docs.google.com/document/d/{document_id}/export?format=docx'

    response = requests.get(url)
    response.raise_for_status()
    
    with io.BytesIO(response.content) as f_in:
        doc = docx.Document(f_in)

    questions = []

    question_heading_style = 'heading 2'
    section_heading_style = 'heading 1'
    
    heading_id = ''
    section_title = ''
    question_title = ''
    answer_text_so_far = ''
    
    for p in doc.paragraphs:
        style = p.style.name.lower()
        p_text = clean_line(p.text)
    
        if len(p_text) == 0:
            continue
    
        if style == section_heading_style:
            section_title = p_text
            continue
    
        if style == question_heading_style:
            answer_text_so_far = answer_text_so_far.strip()
            if answer_text_so_far != '' and section_title != '' and question_title != '':
                questions.append({
                    'text': answer_text_so_far,
                    'section': section_title,
                    'question': question_title,
                })
                answer_text_so_far = ''
    
            question_title = p_text
            continue
        
        answer_text_so_far += '\n' + p_text
    
    answer_text_so_far = answer_text_so_far.strip()
    if answer_text_so_far != '' and section_title != '' and question_title != '':
        questions.append({
            'text': answer_text_so_far,
            'section': section_title,
            'question': question_title,
        })

    return questions

In [100]:
faq_documents_ids = {
    'llm-zoomcamp': '1qZjwHkvP0lXHiE4zdbWyUXSVfmVGzougDD6N37bat3E',
}

In [101]:
def load_data():
    faq_documents = []
    for course, course_doc_id in faq_documents_ids.items():
        faq_documents.append({'course': course, 'documents': fetching_faq_document(course_doc_id)})

    return faq_documents

In [102]:

def generate_document_id(doc):
    combined = f"{doc['course']}-{doc['question']}-{doc['text'][:10]}"
    hash_object = hashlib.md5(combined.encode())
    hash_hex = hash_object.hexdigest()
    document_id = hash_hex[:8]
    return document_id

def transform(data):
    encoded_documents = []

    for document in data['documents']:
        document['course'] = data['course']
        document['document_id'] = generate_document_id(document)
        encoded_documents.append(document)

    return encoded_documents

In [103]:
data = transform(load_data()[0])

In [105]:
for idx, item in enumerate(data):
  if item["document_id"] == "a976d6e7":
    print(idx)
    x = item
    break

24


In [106]:
x

{'text': 'Prior to using Ollama models in llm-zoomcamp tasks, you need to have ollama installed on your pc and the relevant LLM model downloaded with ollama from https://www.ollama.com\nTo download ollama for Ubuntu:\n``` curl -fsSL https://ollama.com/install.sh | sh ```\nTo download ollama for Mac and Windows, follow the guide on this link:\nhttps://ollama.com/download/\nOllama a number of open-source LLMs like:\nLlama3\nPhi3\nMistral and Mixtral\nGemma\nQwen\nYou can explore more models on https://ollama.com/library/\nTo download a model in Ollama, simply open command prompt and type:\n``` ollama run model_name ```\ne.g.\n``` ollama run phi3 ```\nIt will automatically download the model and you can use it same way as above for later time.\nTo use Ollama models for inference and llm-zoomcamp tasks, use the following function:\nimport ollama\ndef llm(prompt):\nresponse = ollama.chat(\nmodel="llama3",\nmessages=[{"role": "user", "content": prompt}]\n)\nreturn response[\'message\'][\'con

In [91]:

connection_string = 'http://localhost:9200'
index_name_prefix = 'documents'
current_time = datetime.now().strftime("%Y%m%d_%M%S")
index_name = f"{index_name_prefix}_{current_time}"
number_of_shards = 1
number_of_replicas = 0

es_client = Elasticsearch(connection_string)

print(f'Connecting to Elasticsearch at {connection_string}')

index_settings = {
    "settings": {
        "number_of_shards": number_of_shards,
        "number_of_replicas": number_of_replicas
    },
    "mappings": {
        "properties": {
            "text": {"type": "text"},
            "section": {"type": "text"},
            "question": {"type": "text"},
            "course": {"type": "keyword"},
            "document_id": {"type": "keyword"}
        }
    }
}

print(f'Indexing {len(data)} documents to Elasticsearch index {index_name}')
for document in data:
    print(f'Indexing document {document["document_id"]}')

    es_client.index(index=index_name, document=document)

print(document)

Connecting to Elasticsearch at http://localhost:9200
Indexing 86 documents to Elasticsearch index documents_20240820_5500
Indexing document 97872393
Indexing document a57f9581
Indexing document a5301a1f
Indexing document 1a9b8b53
Indexing document 0536ca0b
Indexing document aace1f4a
Indexing document a1419bf6
Indexing document 258a03fe
Indexing document a705279d
Indexing document fa136280
Indexing document fb81c6ff
Indexing document bf024675
Indexing document e0d2caf7
Indexing document 7bd989aa
Indexing document 1c96a1fb
Indexing document 01cb301e
Indexing document ca68d283
Indexing document 6fc3236a
Indexing document 6d61aae2
Indexing document cbe66cfe
Indexing document 9816f1ae
Indexing document 98c1bc60
Indexing document befeedef
Indexing document baea0a66
Indexing document a976d6e7
Indexing document 18a32cec
Indexing document 764f2789
Indexing document baae926f
Indexing document 190fc999
Indexing document d8c4c7bb
Indexing document a310259a
Indexing document 5a995cf3
Indexing docum

In [88]:
def filter_search(query, size=5, search_words=[]):
    def filter_builder():
        return list(map(lambda word: {"term": {**word}}, search_words))
        
    search_query = {
        "size": size,
        "query": {
            "bool": {
                "must": {
                    "multi_match": {
                        "query": query,
                        "fields": ["question" ,"text"],
                        "type": "best_fields"
                    }
                },
                "filter": filter_builder()
            }
        }
    }

    return es_client.search(index=index_name, body=search_query)

In [89]:
results = filter_search("When is the next cohort?")

In [90]:
results['hits']['hits'][0]

{'_index': 'documents_20240820_5057',
 '_id': 'tXTWbJEB7B8OT7_TJ-Au',
 '_score': 8.443945,
 '_source': {'text': 'Summer 2025 (via Alexey).',
  'section': 'General course-related questions',
  'question': 'When will the course be offered next?',
  'course': 'llm-zoomcamp',
  'document_id': 'bf024675'}}

In [67]:
for index in es_client.indices.get_alias(index="*"):
  es_client.indices.delete(index=index)